In [2]:

# Parte 2 - VulnerabilidadesIAdata
# Comprobación inicial de Python

import sys

print("Proyecto: VulnerabilidadesIAdata")
print("Versión de Python:", sys.version.split()[0])
print("Entorno preparado")


Proyecto: VulnerabilidadesIAdata
Versión de Python: 3.14.2
Entorno preparado



# VulnerabilidadesIAdata
## Parte 2 — Análisis y predicción de vulnerabilidades CVE

**Dataset:** NVD CVE-2025 (NIST)

## 1. Resumen ejecutivo

En esta segunda parte de mi proyecto voy a trabajar
con datos reales de vulnerabilidades informáticas
para intentar predecir qué nivel de gravedad
pueden tener.

La idea es ayudar a las empresas a saber qué
vulnerabilidades deberían revisar primero,
ya que no todas tienen la misma importancia
ni tienen el mismo riesgo.

Para ello, voy a analizar los datos, crear
gráficos y probar diferentes modelos de
Machine Learning para ver cuál funciona mejor.

También voy a comparar los resultados para
comprobar si estos modelos pueden ayudar
a tomar decisiones en ciberseguridad.

Y al final, explicaré los resultados que he
obtenido, los posibles errores del modelo
y cómo se podría utilizar en una empresa.



## 2. Del problema al Machine Learning

En la primera parte expliqué cómo
la inteligencia artificial podría ayudar a las
empresas a gestionar mejor las vulnerabilidades.

Ahora quiero comprobar si es posible
predecir la gravedad de una vulnerabilidad
utilizando la información que tenemos sobre ella.

Voy a utilizar el dataset NVD CVE-2025,
que contiene información sobre vulnerabilidades
informáticas.

Mi objetivo es crear un modelo
que pueda distinguir entre vulnerabilidades
de gravedad baja, media, alta y crítica.

Esto podría ayudar a los responsables de
ciberseguridad a organizar mejor su trabajo
y decidir qué vulnerabilidades revisar primero.

La predicción será una ayuda para tomar decisiones,
pero no sustituirá la revisión de una mano humana.



### 1.1. Definición del modelo

| Pregunta | Mi proyecto |
|---|---|
| ¿Qué decisión quiero hacer mejor? | Ayudar a decidir qué vulnerabilidades revisar primero. |
| ¿Qué quiero predecir? | La gravedad de una vulnerabilidad: baja, media, alta o crítica. |
| ¿Qué representa cada registro? | Una vulnerabilidad identificada mediante un código CVE. |
| ¿Cuándo se realizará la predicción? | Cuando se registre una vulnerabilidad y todavía no tenga una clasificación definitiva. |
| ¿Qué información utilizaré? | La descripción de la vulnerabilidad y otras características disponibles en el momento. |
| ¿Cómo mediré los resultados? | Con métricas como precision, recall, F1-score... |
| ¿Qué beneficio espero conseguir? | Ayudar a los equipos de seguridad a identificar antes las vulnerabilidades más importantes. |



## 3. Datos y calidad del dataset

Ahora voy a analizar los datos que
utilizaré para entrenar mi modelo de Machine Learning.

Mi dataset viene de de la National Vulnerability
Database (NVD) y contiene información sobre
vulnerabilidades informáticas registradas con
identificadores CVE.

El archivo original contiene 45.373 registros.

Antes de entrenar el modelo, voy a comprobar
qué información tiene cada vulnerabilidad,
si hay datos que faltan y si existen registros
dobles osea duplicados.

También revisaré cuáles tienen una gravedad
asignada y cuáles no.

Este análisis me ayudará a preparar los datos
y evitar problemas que puedan afectar
a las predicciones.


In [3]:

# Cargar el dataset original de vulnerabilidades CVE

import json
from pathlib import Path

ruta = Path("data/nvdcve-2.0-2025.json")

with open(ruta, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

print("Dataset cargado correctamente")
print("Total de vulnerabilidades:", len(datos["vulnerabilities"]))
print("Formato:", datos["format"])
print("Versión:", datos["version"])


Dataset cargado correctamente
Total de vulnerabilidades: 45373
Formato: NVD_CVE
Versión: 2.0


In [4]:

# Comprobar duplicados y clasificaciones

import pandas as pd

registros = []

for elemento in datos["vulnerabilities"]:
    cve = elemento["cve"]

    registros.append({
        "cve_id": cve.get("id"),
        "fecha": cve.get("published"),
        "estado": cve.get("vulnStatus"),
        "tiene_cvss": any(
            clave.startswith("cvssMetric")
            for clave in cve.get("metrics", {})
        )
    })

df = pd.DataFrame(registros)

print("Total de registros:", len(df))
print("CVE duplicados:", df["cve_id"].duplicated().sum())
print("Registros con métricas CVSS:", df["tiene_cvss"].sum())
print("Registros sin métricas CVSS:", (~df["tiene_cvss"]).sum())

display(df.head())


Total de registros: 45373
CVE duplicados: 0
Registros con métricas CVSS: 43176
Registros sin métricas CVSS: 2197


,cve_id,fecha,estado,tiene_cvss
0,CVE-2025-0168,2025-01-01T14:15:23.590,Analyzed,True
1,CVE-2025-22214,2025-01-02T04:15:06.277,Deferred,True
2,CVE-2025-0171,2025-01-02T15:15:25.550,Analyzed,True
3,CVE-2025-0172,2025-01-02T16:15:09.103,Analyzed,True
4,CVE-2025-0173,2025-01-02T18:15:21.630,Analyzed,True


In [5]:

# Comprobar las librerías que se necesitan para el proyecto

import json
import pandas as pd
import numpy as np
import plotly.express as px
import sklearn
import joblib

print("Todas las librerías se han cargado correctamente")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)


Todas las librerías se han cargado correctamente
Pandas: 3.1.1
NumPy: 2.5.5
Scikit-learn: 1.9.1



### 3.1 Primeros resultados del análisis

Después de cargar el dataset,
contiene 45.373 vulnerabilidades.

De todas ellas, 43.176 tienen alguna métrica
CVSS y 2.197 no tienen esta información.

Esto significa que mas o menos el 95,16 %
de los registros cuentan con datos de evaluación
CVSS, mientras que al 4,84 % les falta.

También he comprobado que no hay identificadores
CVE duplicados, por lo tanto es positivo porque
evita contar dos veces la misma vulnerabilidad.

Esto me ayuda a conocer mejor
la calidad de los datos antes de entrenar
los modelos.

### ¿Qué significa esto para una empresa?

Para una empresa pues es necesario el saber qué
vulnerabilidades tienen información suficiente
para poder analizarlas.

Las que no tienen una evaluación CVSS necesitarán
una revisión adicional, ya que no podemos
dar por hecho su nivel de gravedad.


In [6]:

# Gráfico de disponibilidad de métricas CVSS

resumen_cvss = (
    df["tiene_cvss"]
    .map({True: "Con CVSS", False: "Sin CVSS"})
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="Cantidad")
)

fig = px.bar(
    resumen_cvss,
    x="Estado",
    y="Cantidad",
    color="Estado",
    text="Cantidad",
    title="Disponibilidad de métricas CVSS en el dataset",
    labels={
        "Estado": "Disponibilidad de CVSS",
        "Cantidad": "Número de vulnerabilidades"
    }
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()


In [7]:

# Ver la gravedad de las vulnerabilidades

def obtener_gravedad(cve):
    metricas = cve.get("metrics", {})

    # Priorizar la evaluación principal de NVD,
    # cuando esté disponible.
    for version in [
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV40"
    ]:
        evaluaciones = metricas.get(version, [])

        if evaluaciones:
            principal = next(
                (
                    m for m in evaluaciones
                    if m.get("type") == "Primary"
                    and m.get("source") == "nvd@nist.gov"
                ),
                evaluaciones[0]
            )

            return principal.get(
                "cvssData", {}
            ).get("baseSeverity")

    return None


# Extraer la gravedad de todos los registros
df["gravedad"] = [
    obtener_gravedad(item["cve"])
    for item in datos["vulnerabilities"]
]

print("Distribución de vulnerabilidades:")
print(df["gravedad"].fillna("SIN CLASIFICAR").value_counts())


Distribución de vulnerabilidades:
gravedad
MEDIUM            19678
HIGH              15988
CRITICAL           6304
SIN CLASIFICAR     2197
LOW                1190
NONE                 16
Name: count, dtype: int64


In [8]:

# Gráfico de vulnerabilidades por gravedad

orden = [
    "LOW", "MEDIUM", "HIGH",
    "CRITICAL", "NONE", "SIN CLASIFICAR"
]

conteo = (
    df["gravedad"]
    .fillna("SIN CLASIFICAR")
    .value_counts()
    .reindex(orden, fill_value=0)
    .reset_index()
)

conteo.columns = ["Gravedad", "Cantidad"]

fig = px.bar(
    conteo,
    x="Gravedad",
    y="Cantidad",
    color="Gravedad",
    text="Cantidad",
    title="Distribución de vulnerabilidades por gravedad",
    labels={
        "Gravedad": "Nivel de gravedad",
        "Cantidad": "Número de vulnerabilidades"
    }
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()



### 3.2 Análisis de la gravedad

Después de analizar los datos, he visto que
la mayoría de las vulnerabilidades tienen
una gravedad media o alta.

En concreto, hay 19.678 vulnerabilidades
de gravedad media, 15.988 altas y 6.304 críticas.

También hay 1.190 de gravedad baja,
2.197 sin clasificar y 16 con gravedad NONE.

Me he fijado bastante en que hay muchas
vulnerabilidades altas y críticas, por lo que
es importante que las empresas tengan una forma
de organizar cuáles deben revisar primero.

Estos resultados muestran que una empresa puede
tener muchas vulnerabilidades que revisar.

Así es que este modelo predictivo podría ayudar a detectar
antes las que parecen más graves, pero siempre
sería necesario comprobar el contexto de cada
empresa antes de decidir cuáles solucionar primero.


In [9]:

# Comprobar datos que falten

# Extraer las descripciones en inglés
descripciones = []

for elemento in datos["vulnerabilities"]:
    cve = elemento["cve"]

    descripcion = next(
        (
            d.get("value")
            for d in cve.get("descriptions", [])
            if d.get("lang") == "en"
        ),
        None
    )

    descripciones.append(descripcion)

df["descripcion"] = descripciones

# Comprobar los valores que no estan
columnas = [
    "cve_id",
    "fecha",
    "estado",
    "descripcion",
    "gravedad"
]

faltantes = df[columnas].isna().sum()

print("Datos que faltan por columna:")
print(faltantes)

print("\nTotal de registros:", len(df))


Datos que faltan por columna:
cve_id            0
fecha             0
estado            0
descripcion       0
gravedad       2197
dtype: int64

Total de registros: 45373



### 3.3 Comprobación de datos que no estan

He revisado las columnas del dataset
para ver si faltaba información.

Los resultados muestran que los identificadores CVE,
las fechas, los estados y las descripciones
no tienen valores vacíos.

Sin embargo, hay 2.197 vulnerabilidades que
no tienen una gravedad asignada.

También he encontrado 16 registros con gravedad NONE,
que no forman parte de las cuatro categorías
que quiero predecir.

Por eso, para entrenar los modelos utilizaré
los registros que tengan una gravedad válida:
LOW, MEDIUM, HIGH o CRITICAL.

No voy a inventar cualquier gravedad para los registros
que no la tienen, porque eso podría afectar
a los resultados del modelo.

In [10]:

# Seleccionar las cuatro categorías de gravedad

categorias_validas = [
    "LOW",
    "MEDIUM",
    "HIGH",
    "CRITICAL"
]

# Crear una copia con registros válidos
df_limpio = df[
    df["gravedad"].isin(categorias_validas)
].copy()

# Convertir las fechas a formato fecha
df_limpio["fecha"] = pd.to_datetime(
    df_limpio["fecha"],
    errors="coerce"
)

print("Registros originales:", len(df))
print("Registros preparados:", len(df_limpio))
print("Registros excluidos:", len(df) - len(df_limpio))

print("\nDistribución final:")
print(df_limpio["gravedad"].value_counts())

print("\nFechas no válidas:")
print(df_limpio["fecha"].isna().sum())


Registros originales: 45373
Registros preparados: 43160
Registros excluidos: 2213

Distribución final:
gravedad
MEDIUM      19678
HIGH        15988
CRITICAL     6304
LOW          1190
Name: count, dtype: int64

Fechas no válidas:
0



### 3.4 Limpieza de los datos

Después de revisar los datos, he preparado una
versión del dataset para utilizarla en los modelos.

De los 45.373 registros originales, he conservado 43.160 porque tienen una gravedad válida.

He quitado 2.213 registros: 2.197 que no tenían
una gravedad asignada y 16 que aparecían como NONE.

No he eliminado estos registros del archivo original, solo los he como separado de los datos que voy a utilizar para entrenar el modelo.

También he convertido las fechas al formato adecuado y he comprobado que no hay fechas inválidas.

Con esta limpieza puedo trabajar con datos que
tienen una clasificación conocida y evaluar mejor las predicciones del modelo.

Aun así, las vulnerabilidades sin clasificación
siguen siendo importantes y deberían revisarse
igualmente por separado.


In [11]:

# Analizar las fechas del dataset

df_limpio["anio"] = df_limpio["fecha"].dt.year
df_limpio["mes"] = df_limpio["fecha"].dt.to_period("M").astype(str)

print("Primera fecha:", df_limpio["fecha"].min())
print("Última fecha:", df_limpio["fecha"].max())

print("\nVulnerabilidades por año:")
print(df_limpio["anio"].value_counts().sort_index())

print("\nVulnerabilidades por mes:")
print(df_limpio["mes"].value_counts().sort_index())


Primera fecha: 2025-01-01 14:15:23.590000
Última fecha: 2026-10-15 09:17:14.190000

Vulnerabilidades por año:
anio
2025    37977
2026     5183
Name: count, dtype: int64

Vulnerabilidades por mes:
mes
2025-01    1854
2025-02    1793
2025-03    2921
2025-04    3691
2025-05    3226
2025-06    3209
2025-07    3582
2025-08    3349
2025-09    3730
2025-10    3617
2025-11    2855
2025-12    4150
2026-01    1991
2026-02    1081
2026-03     640
2026-04     271
2026-05     337
2026-06     392
2026-07     161
2026-08     150
2026-09     112
2026-10      48
Name: count, dtype: int64


In [12]:
# Evolución mes a mes de las vulnerabilidades

vulnerabilidades_mes = (
    df_limpio.groupby("mes")
    .size()
    .reset_index(name="cantidad")
    .sort_values("mes")
)

fig = px.line(
    vulnerabilidades_mes,
    x="mes",
    y="cantidad",
    markers=True,
    title="Evolución mensual de vulnerabilidades CVE",
    labels={
        "mes": "Mes de publicación",
        "cantidad": "Número de vulnerabilidades"
    }
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()



### 3.5 Análisis de las fechas

He revisado las fechas de publicación de las
vulnerabilidades para comprobar cómo se
distribuyen a lo largo del tiempo, el dataset contiene registros publicados tanto en 2025 como en 2026, aunque el nombre del archivo haga referencia a 2025.

Con el gráfico puedo ver cómo cambia la cantidad de vulnerabilidades publicadas cada mes.

Esto es importante porque, cuando entrene
los modelos, utilizaré los datos más antiguos
para entrenar y los más recientes para comprobar si las predicciones funcionan.

Las vulnerabilidades nuevas aparecen continuamente, por lo que una empresa necesita cosas que puedan analizar información reciente.

Por eso quiero comprobar que mi modelo funciona
también con vulnerabilidades pasadas a las
que utilizó para aprender.



## 4. Qué he descubierto al analizar los datos

Me interesa comprobar cuáles son los tipos
de debilidades más frecuentes y si pueden
tener alguna relación con la gravedad.

Esto también me ayudará a decidir qué información podría utilizar para entrenar los modelos de Machine Learning.


In [13]:

# Extraer los códigos CWE de las vulnerabilidades

def obtener_cwe(cve):
    codigos = []

    for debilidad in cve.get("weaknesses", []):
        for descripcion in debilidad.get("description", []):
            valor = descripcion.get("value", "")

            if valor.startswith("CWE-") and valor not in codigos:
                codigos.append(valor)

    return codigos[0] if codigos else "SIN CWE"


df["cwe"] = [
    obtener_cwe(elemento["cve"])
    for elemento in datos["vulnerabilities"]
]

df_limpio["cwe"] = df.loc[df_limpio.index, "cwe"]

print("Los 10 códigos CWE más frecuentes:")
print(df_limpio["cwe"].value_counts().head(10))


Los 10 códigos CWE más frecuentes:
cwe
CWE-79     7611
CWE-74     2566
CWE-862    2234
CWE-352    1720
CWE-89     1567
SIN CWE    1351
CWE-119    1153
CWE-22     1040
CWE-284     993
CWE-125     831
Name: count, dtype: int64


In [14]:

# Gráfico de los 10 CWE más frecuentes

top_cwe = (
    df_limpio["cwe"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_cwe.columns = ["CWE", "Cantidad"]

fig = px.bar(
    top_cwe,
    x="CWE",
    y="Cantidad",
    color="CWE",
    text="Cantidad",
    title="Los 10 tipos de debilidades más frecuentes",
    labels={
        "CWE": "Código de debilidad",
        "Cantidad": "Número de vulnerabilidades"
    }
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()



### 4.1 Tipos de debilidades más frecuentes

Después de analizar los códigos CWE, he visto que el más frecuente es CWE-79, relacionado con
vulnerabilidades Cross-Site Scripting (XSS),
con 7.611 registros.

También aparecen otros tipos de problemas,
como las inyecciones, los fallos de autorización y los errores de control de acceso.

Esto me ha ayudado a entender que no todas las
vulnerabilidades son iguales y que pueden
afectar a diferentes partes de un sistema.

Para este análisis he utilizado el primer código CWE disponible en cada registro, aunque algunas vulnerabilidades pueden tener varios.

In [15]:

# Relación entre los tipos CWE y la gravedad

principales_cwe = (
    df_limpio["cwe"]
    .value_counts()
    .head(10)
    .index
)

datos_cwe = df_limpio[
    df_limpio["cwe"].isin(principales_cwe)
]

tabla_cwe_gravedad = pd.crosstab(
    datos_cwe["cwe"],
    datos_cwe["gravedad"],
    normalize="index"
) * 100

tabla_cwe_gravedad = tabla_cwe_gravedad.reindex(
    columns=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
    fill_value=0
)

display(tabla_cwe_gravedad.round(2))


gravedad,LOW,MEDIUM,HIGH,CRITICAL
cwe,,,,
CWE-119,3.21,12.06,60.02,24.72
CWE-125,4.33,40.55,51.14,3.97
CWE-22,2.50,37.79,43.17,16.54
CWE-284,3.02,37.66,39.48,19.84
CWE-352,0.70,56.45,41.22,1.63
CWE-74,0.23,5.38,30.67,63.72
CWE-79,2.15,76.93,20.01,0.91
CWE-862,1.30,76.19,18.62,3.89
CWE-89,0.51,20.55,46.84,32.10


In [16]:

# Gráfico de gravedad por tipo de CWE

tabla_grafico = (
    tabla_cwe_gravedad
    .reset_index()
    .melt(
        id_vars="cwe",
        var_name="Gravedad",
        value_name="Porcentaje"
    )
)

fig = px.bar(
    tabla_grafico,
    x="cwe",
    y="Porcentaje",
    color="Gravedad",
    barmode="stack",
    title="Distribución de gravedad según el tipo de CWE",
    labels={
        "cwe": "Tipo de debilidad CWE",
        "Porcentaje": "Porcentaje de vulnerabilidades (%)"
    },
    category_orders={
        "Gravedad": ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_range=[0, 100]
)

fig.show()



### 4.2 Relación entre las debilidades y la gravedad

He comparado los tipos de debilidades CWE con
los niveles de gravedad de las vulnerabilidades.

Los resultados muestran que existen diferencias
bastante grandes según el tipo de problema.

Por ejemplo, el 63,72 % de las vulnerabilidades
CWE-74 son críticas, mientras que en CWE-79
la mayoría son de gravedad media (76,93 %).

También he visto que el 32,10 % de las
vulnerabilidades CWE-89 son críticas.

Esto me hace pensar que el tipo de debilidad
podría ser una variable útil para mi modelo
de Machine Learning.

Sin embargo, todavía tengo que comprobarlo
entrenando los modelos y comparando sus resultados.

Aun así, no se debe decidir la prioridad
de una vulnerabilidad solo por su CWE,
porque también influyen otros factores,
como los sistemas afectados y otros motivos.



### 4.3 Comprobación de las hipótesis

En la primera parte de mi proyecto planteé
varias hipótesis sobre las vulnerabilidades
y el uso de Machine Learning.

Ahora voy a comprobarlas utilizando los
datos reales del dataset.

**Hipótesis 1: Relación entre el vector de ataque y la gravedad**

Quiero comprobar si las vulnerabilidades que
se pueden explotar a través de la red suelen
tener una gravedad mayor que las que necesitan
otro tipo de acceso.

Para ello, voy a comparar los niveles de gravedad según el vector de ataque de cada vulnerabilidad.


In [17]:

# Hipótesis 1: vector de ataque y gravedad

def obtener_vector_ataque(cve):
    metricas = cve.get("metrics", {})

    for version in [
        "cvssMetricV31",
        "cvssMetricV30",
        "cvssMetricV40"
    ]:
        evaluaciones = metricas.get(version, [])

        if evaluaciones:
            principal = next(
                (
                    m for m in evaluaciones
                    if m.get("type") == "Primary"
                    and m.get("source") == "nvd@nist.gov"
                ),
                evaluaciones[0]
            )

            return principal.get(
                "cvssData", {}
            ).get("attackVector")

    return None


# Añadir el vector de ataque
df["vector_ataque"] = [
    obtener_vector_ataque(item["cve"])
    for item in datos["vulnerabilities"]
]

df_limpio["vector_ataque"] = df.loc[
    df_limpio.index, "vector_ataque"
]

# Calcular porcentaje de vulnerabilidades altas y críticas
tabla_vector = pd.crosstab(
    df_limpio["vector_ataque"],
    df_limpio["gravedad"],
    normalize="index"
) * 100

tabla_vector = tabla_vector.reindex(
    columns=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
    fill_value=0
)

tabla_vector["HIGH + CRITICAL (%)"] = (
    tabla_vector["HIGH"] + tabla_vector["CRITICAL"]
)

display(tabla_vector.round(2))


gravedad,LOW,MEDIUM,HIGH,CRITICAL,HIGH + CRITICAL (%)
vector_ataque,,,,,
ADJACENT,3.90,34.42,43.51,18.18,61.69
ADJACENT_NETWORK,8.84,48.88,40.27,2.01,42.28
LOCAL,4.47,47.69,47.39,0.46,47.85
NETWORK,2.03,44.63,34.70,18.64,53.34
PHYSICAL,13.37,77.80,8.83,0.00,8.83


In [18]:

# Unificar los nombres de vectores

df_limpio["vector_ataque_unificado"] = (
    df_limpio["vector_ataque"]
    .replace({
        "ADJACENT_NETWORK": "ADJACENT"
    })
)

# Calcular cuántas vulnerabilidades hay en cada grupo
resumen_vector = (
    df_limpio
    .dropna(subset=["vector_ataque_unificado"])
    .groupby("vector_ataque_unificado")
    .agg(
        total=("gravedad", "size"),
        altas_criticas=(
            "gravedad",
            lambda x: x.isin(["HIGH", "CRITICAL"]).sum()
        )
    )
)

resumen_vector["porcentaje"] = (
    resumen_vector["altas_criticas"]
    / resumen_vector["total"] * 100
)

display(resumen_vector.round(2))


,total,altas_criticas,porcentaje
vector_ataque_unificado,,,
ADJACENT,1048,473,45.13
LOCAL,8331,3986,47.85
NETWORK,33362,17796,53.34
PHYSICAL,419,37,8.83



### 4.4 Resultado de la hipótesis 1

Después de comparar los diferentes vectores
de ataque, he visto que las vulnerabilidades
que se pueden explotar por la red tienen
un porcentaje más alto de gravedad alta o crítica.

En concreto, el 53,34 % de las vulnerabilidades
de tipo NETWORK son altas o críticas.

En comparación, las de tipo LOCAL tienen
un 47,85 %, las ADJACENT un 45,13 % y
las PHYSICAL un 8,83 %.

Aun así, esto no significa que todas las
vulnerabilidades de red sean más graves,
sino que en este dataset aparecen con
una mayor proporción de gravedad alta o crítica.


### 4.5 Resumen de las hipótesis

| Hipótesis | Estado | Evidencia |
|---|---|---|
| H1. Vector de ataque y gravedad | Respaldada | NETWORK tiene un 53,34 % de vulnerabilidades altas o críticas, por encima de los demás vectores. |
| H2. Tipo de debilidad y gravedad | Respaldada | CWE-74 tiene un 63,72 % de vulnerabilidades críticas, frente al 0,91 % de CWE-79. |
| H3. Clasificación mediante IA | Pendiente | Todavía tengo que entrenar los modelos y comparar sus resultados. |
| H4. Indicadores de rendimiento | Parcialmente evaluada | Ya he calculado la cantidad de vulnerabilidades y la cobertura CVSS, pero faltan las métricas de los modelos. |



## 5. Variables que voy a utilizar en el modelo

En este apartado voy a seleccionar la información que utilizará mi modelo de Machine Learning para intentar predecir la gravedad de las vulnerabilidades.

Para empezar, utilizaré principalmente la
descripción de cada vulnerabilidad, ya que
contiene información sobre el fallo de seguridad.

También estudiaré si el tipo de debilidad
(CWE) puede mejorar las predicciones.

Sin embargo, tengo que comprobar si esa información estaba disponible en el momento de la predicción.

La idea es que una empresa pueda obtener una
estimación inicial de gravedad a partir de
la información disponible sobre una
vulnerabilidad nueva.

Esta predicción serviría como apoyo mientras
se realiza una evaluación de seguridad más
completa.


In [19]:

# Analizar las descripciones de las vulnerabilidades

df_limpio["longitud_descripcion"] = (
    df_limpio["descripcion"].str.len()
)

print("Longitud media de las descripciones:")
print(round(df_limpio["longitud_descripcion"].mean(), 2))

print("\nDescripción más corta:")
print(df_limpio["longitud_descripcion"].min())

print("\nDescripción más larga:")
print(df_limpio["longitud_descripcion"].max())

print("\nPrimeros ejemplos:")
display(
    df_limpio[
        ["cve_id", "descripcion", "gravedad"]
    ].head(5)
)


Longitud media de las descripciones:
388.14

Descripción más corta:
15

Descripción más larga:
3998

Primeros ejemplos:


,cve_id,descripcion,gravedad
0,CVE-2025-0168,A vulnerability classified as critical has bee...,HIGH
1,CVE-2025-22214,Landray EIS 2001 through 2006 allows Message/f...,MEDIUM
2,CVE-2025-0171,"A vulnerability, which was classified as criti...",HIGH
3,CVE-2025-0172,A vulnerability has been found in code-project...,HIGH
4,CVE-2025-0173,A vulnerability was found in SourceCodester On...,HIGH


In [20]:

# Buscar palabras que podrían revelar la gravedad

patron = (
    r"\b(critical|high severity|medium severity|"
    r"low severity|cvss score|cvss base score|"
    r"cvss:3\.[01]|cvss:4\.0)\b"
)

df_limpio["texto_con_gravedad"] = (
    df_limpio["descripcion"]
    .str.contains(patron, case=False, regex=True, na=False)
)

total = df_limpio["texto_con_gravedad"].sum()

print("Descripciones con posibles pistas de gravedad:", total)
print("Porcentaje:", round(total / len(df_limpio) * 100, 2), "%")


/tmp/ipykernel_2930/227860728.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(patron, case=False, regex=True, na=False)


Descripciones con posibles pistas de gravedad: 3282
Porcentaje: 7.6 %



### 5.1 Revisión de las descripciones

He revisado las descripciones de las vulnerabilidades para comprobar si contienen palabras que puedan dar pistas sobre su gravedad.

He encontrado 3.282 descripciones con expresiones relacionadas con niveles de gravedad o CVSS, lo que representa aproximadamente el 7,60 % de los registros preparados.

Esto es importante porque no quiero que mi modelo aprenda simplemente a reconocer palabras como
"critical" o puntuaciones CVSS.

Mi objetivo es que aprenda a partir de la
información técnica de las vulnerabilidades.


In [21]:

# Preparar descripciones
import re

patron_cvss = (
    r"\b(?:critical|high severity|medium severity|"
    r"low severity|cvss score|cvss base score)\b"
    r"|CVSS:[234]\.[0-9](?:/[A-Za-z0-9_:.-]+)*"
)

def limpiar_descripcion(texto):
    if not isinstance(texto, str):
        return ""

    texto = re.sub(
        patron_cvss,
        " ",
        texto,
        flags=re.IGNORECASE
    )

    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


df_limpio["descripcion_limpia"] = (
    df_limpio["descripcion"].apply(limpiar_descripcion)
)

print("Descripciones preparadas:", len(df_limpio))
print("Ejemplo original:")
print(df_limpio["descripcion"].iloc[0][:250])

print("\nEjemplo preparado:")
print(df_limpio["descripcion_limpia"].iloc[0][:250])


Descripciones preparadas: 43160
Ejemplo original:
A vulnerability classified as critical has been found in code-projects Job Recruitment 1.0. This affects an unknown part of the file /_parse/_feedback_system.php. The manipulation of the argument person leads to sql injection. It is possible to initi

Ejemplo preparado:
A vulnerability classified as has been found in code-projects Job Recruitment 1.0. This affects an unknown part of the file /_parse/_feedback_system.php. The manipulation of the argument person leads to sql injection. It is possible to initiate the a


In [22]:

# Comprobar el resultado de la limpieza

modificadas = (
    df_limpio["descripcion"]
    != df_limpio["descripcion_limpia"]
).sum()

vacias = (
    df_limpio["descripcion_limpia"]
    .str.strip()
    .eq("")
    .sum()
)

print("Total de descripciones:", len(df_limpio))
print("Descripciones modificadas:", modificadas)
print("Descripciones vacías:", vacias)

print("\nEjemplos de descripciones:")
display(
    df_limpio[
        ["cve_id", "descripcion_limpia"]
    ].head(5)
)


Total de descripciones: 43160
Descripciones modificadas: 10603
Descripciones vacías: 0

Ejemplos de descripciones:


,cve_id,descripcion_limpia
0,CVE-2025-0168,A vulnerability classified as has been found i...
1,CVE-2025-22214,Landray EIS 2001 through 2006 allows Message/f...
2,CVE-2025-0171,"A vulnerability, which was classified as , was..."
3,CVE-2025-0172,A vulnerability has been found in code-project...
4,CVE-2025-0173,A vulnerability was found in SourceCodester On...



### 5.2 Resultado de la limpieza del texto

Después de preparar las descripciones, he comprobado que sigo teniendo 43.160 registros.

También he comprobado que ninguna descripción
ha quedado vacía después de la limpieza.

He realizado esta preparación para intentar
evitar que el modelo utilice referencias
directas a la gravedad CVSS.

Aun así, esta primera limpieza no garantiza
que se hayan eliminado todas las posibles
pistas sobre la clasificación.



### 4.3. Variables seleccionadas

| Variable | Cómo la voy a utilizar | Por qué puede ayudar |
|---|---|---|
| Descripción limpia | Convertiré el texto en números utilizando TF-IDF. | Puede contener información sobre el fallo de seguridad. |
| CWE | Lo estudiaré como una variable adicional, si se confirma que estaba disponible a tiempo. | Permite identificar el tipo de debilidad. |
| Fecha de publicación | La utilizaré para ordenar y dividir el dataset. | Evita entrenar con información posterior a los datos de prueba. |
| Gravedad CVSS | Será la variable objetivo. | Es la categoría que quiero predecir. |

No voy a utilizar directamente la puntuación
CVSS ni sus componentes para entrenar,
porque podrían revelar la respuesta.

Primero voy a probar modelos basados en
la descripción de las vulnerabilidades.



## 6. División de los datos y modelo de referencia

Antes de entrenar los modelos, voy a dividir
los datos en tres grupos diferentes.

Utilizaré el 70 % para entrenamiento,
el 15 % para validación y el 15 % para prueba.

Como mi dataset tiene fechas de publicación,
voy a ordenar las vulnerabilidades desde
las más antiguas hasta las más recientes.

De esta forma, el modelo aprenderá con
vulnerabilidades pasadas y después
comprobaré cómo funciona con otras posteriores.

También crearé un modelo de referencia
sencillo para comparar sus resultados con
los modelos de Machine Learning.

In [23]:

# Dividir los datos respetando el orden temporal

df_ordenado = (
    df_limpio
    .sort_values(["fecha", "cve_id"])
    .reset_index(drop=True)
)

total = len(df_ordenado)

fin_train = int(total * 0.70)
fin_val = int(total * 0.85)

train = df_ordenado.iloc[:fin_train].copy()
val = df_ordenado.iloc[fin_train:fin_val].copy()
test = df_ordenado.iloc[fin_val:].copy()

print("Entrenamiento:", len(train))
print("Validación:", len(val))
print("Prueba:", len(test))
print("Total:", len(train) + len(val) + len(test))

print("\nFechas de cada grupo:")
for nombre, grupo in [
    ("Entrenamiento", train),
    ("Validación", val),
    ("Prueba", test)
]:
    print(
        nombre,
        "| Desde:", grupo["fecha"].min(),
        "| Hasta:", grupo["fecha"].max()
    )

print("\nDistribución de gravedad por grupo (%):")
distribucion = pd.DataFrame({
    "Entrenamiento": train["gravedad"].value_counts(normalize=True) * 100,
    "Validación": val["gravedad"].value_counts(normalize=True) * 100,
    "Prueba": test["gravedad"].value_counts(normalize=True) * 100
}).fillna(0)

display(distribucion.round(2))


Entrenamiento: 30211
Validación: 6475
Prueba: 6474
Total: 43160

Fechas de cada grupo:
Entrenamiento | Desde: 2025-01-01 14:15:23.590000 | Hasta: 2025-10-27 02:15:49.457000
Validación | Desde: 2025-10-27 02:15:49.607000 | Hasta: 2025-12-18 08:16:09.343000
Prueba | Desde: 2025-12-18 08:16:09.473000 | Hasta: 2026-10-15 09:17:14.190000

Distribución de gravedad por grupo (%):


,Entrenamiento,Validación,Prueba
gravedad,,,
MEDIUM,45.45,48.05,43.79
HIGH,36.50,36.79,39.84
CRITICAL,15.32,12.53,13.38
LOW,2.73,2.64,3.00


In [24]:

# Comprobar que la división temporal es correcta

print("¿Entrenamiento termina antes que validación?")
print(train["fecha"].max() < val["fecha"].min())

print("\n¿Validación termina antes que prueba?")
print(val["fecha"].max() < test["fecha"].min())

print("\nCVE repetidos entre conjuntos:")
print(
    len(
        set(train["cve_id"])
        & set(val["cve_id"])
    )
    +
    len(
        set(train["cve_id"])
        & set(test["cve_id"])
    )
    +
    len(
        set(val["cve_id"])
        & set(test["cve_id"])
    )
)


¿Entrenamiento termina antes que validación?
True

¿Validación termina antes que prueba?
True

CVE repetidos entre conjuntos:
0



### 6.1 Modelo de referencia

Este modelo siempre va a predecir la gravedad
que más aparece en los datos de entrenamiento.

Lo utilizaré para comprobar si los modelos
de inteligencia artificial consiguen
resultados mejores.

Si un modelo más avanzado no supera este
dicho resultado, no tendría mucho sentido
utilizarlo en una empresa.

Este método me permitirá comprobar si
realmente merece la pena utilizar
Machine Learning para ayudar a organizar
las vulnerabilidades.


In [25]:

# Modelo de referencia: predecir siempre
# la clase que más pasa dentro del entrenamiento

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score
)

# Preparar datos
X_train = train[["descripcion_limpia"]]
y_train = train["gravedad"]

X_val = val[["descripcion_limpia"]]
y_val = val["gravedad"]

# Crear y entrenar el baseline
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

# Predicciones sobre validación
pred_baseline = baseline.predict(X_val)

# Evaluación
accuracy = accuracy_score(y_val, pred_baseline)
f1 = f1_score(
    y_val,
    pred_baseline,
    average="macro",
    zero_division=0
)
recall_criticas = recall_score(
    y_val == "CRITICAL",
    pred_baseline == "CRITICAL",
    zero_division=0
)

print("Clase más frecuente:", y_train.mode()[0])
print("Accuracy:", round(accuracy, 4))
print("F1-macro:", round(f1, 4))
print("Recall de críticas:", round(recall_criticas, 4))


Clase más frecuente: MEDIUM
Accuracy: 0.4805
F1-macro: 0.1623
Recall de críticas: 0.0



### 6.2 Resultados del modelo de referencia

He creado un modelo que siempre predice la gravedad más frecuente, que es MEDIUM.

Al mirarlo bien con los datos de validación,
he obtenido un accuracy del 48,05 %
y un F1-macro de 0,1623.

Sin embargo, el recall de las vulnerabilidades
críticas ha sido del 0 %.

Esto quiere decir que pues el modelo acierta algunas vulnerabilidades, pero no consigue identificar ninguna de las críticas.

Por eso, aunque su porcentaje de aciertos
pueda parecer aceptable, no sería una buena
opción para utilizarlo en una empresa.

Un sistema que no identifica vulnerabilidades
críticas podría hacer que la parte de seguridad dejara sin revisar problemas bastante importantes.


## 7. Modelos de Machine Learning

### 7.1 Primer modelo: Logística

Ahora voy a entrenar mi primer modelo utilizando las descripciones de las vulnerabilidades.

Para que el modelo pueda entender los textos,
voy a utilizar TF-IDF, que convierte las palabras en valores numéricos.

Después utilizaré una Regresión Logística para
intentar ordenar las vulnerabilidades según
su gravedad.

He elegido este modelo porque es algo
sencillo y me deja ver si la información de las descripciones sirve para realizar predicciones mejores que el baseline.

In [26]:

# Primer modelo: TF-IDF + Regresión Logística

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score

modelo_logistico = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        min_df=3,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Entrenar solamente con los datos de entrenamiento
modelo_logistico.fit(
    train["descripcion_limpia"],
    train["gravedad"]
)

# Ordenar con los datos de validación
pred_logistico = modelo_logistico.predict(
    val["descripcion_limpia"]
)

print("Accuracy:",
      round(accuracy_score(val["gravedad"], pred_logistico), 4))

print("F1-macro:",
      round(f1_score(
          val["gravedad"],
          pred_logistico,
          average="macro",
          zero_division=0
      ), 4))

print("Recall de críticas:",
      round(recall_score(
          val["gravedad"] == "CRITICAL",
          pred_logistico == "CRITICAL",
          zero_division=0
      ), 4))


Accuracy: 0.6703
F1-macro: 0.4775
Recall de críticas: 0.4636



### 7.2 Resultados de la Regresión Logística

Después de entrenar mi primer modelo de
Machine Learning, he obtenido un accuracy
del 67,03 % y un F1-macro de 0,4775.

Estos resultados son mucho mejoes que los del
modelo de referencia, que tenía un accuracy
del 48,05 %.

También he conseguido que el modelo detecte
el 46,36 % de las vulnerabilidades críticas,
mientras que el baseline no detectaba ninguna.

Esto muestra que las descripciones de las
vulnerabilidades contienen información que
puede ayudar a predecir su gravedad.

Aun así, el modelo todavía no identifica
más de la mitad de las vulnerabilidades críticas, osea que se necesita mejorar.

Este modelo podría ayudar a organizar mejor
las vulnerabilidades que tiene que revisar
un equipo de seguridad, aunque todavía no se debereía usar como único sistema de prioridad, porque podría olvidarse de vulnerabilidades importantes.


### 7.3 Segundo modelo: Random Forest

Ahora voy a probar un segundo modelo llamado
Random Forest.

Este algoritmo utiliza varios pasos de
decisión para aprender patrones en los datos.

Quiero comprobar si consigue mejores resultados que la Regresión Logística, especialmente al identificar vulnerabilidades críticas.

Después compararé ambos modelos utilizando
las mismas métricas y los mismos datos
de validación.

In [27]:

# Segundo modelo: TF-IDF + Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, recall_score

modelo_rf = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=3000,
        min_df=3,
        ngram_range=(1, 1),
        stop_words="english"
    )),
    ("clasificador", RandomForestClassifier(
        n_estimators=60,
        max_depth=18,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=2
    ))
])

# Entrenar con los datos de el entrenamiento
modelo_rf.fit(
    train["descripcion_limpia"],
    train["gravedad"]
)

# Predecir los datos de validación
pred_rf = modelo_rf.predict(
    val["descripcion_limpia"]
)

# Calcular las métricas
accuracy_rf = accuracy_score(
    val["gravedad"], pred_rf
)

f1_rf = f1_score(
    val["gravedad"],
    pred_rf,
    average="macro",
    zero_division=0
)

recall_criticas_rf = recall_score(
    val["gravedad"] == "CRITICAL",
    pred_rf == "CRITICAL",
    zero_division=0
)

print("Accuracy:", round(accuracy_rf, 4))
print("F1-macro:", round(f1_rf, 4))
print("Recall de críticas:", round(recall_criticas_rf, 4))


Accuracy: 0.6483
F1-macro: 0.4162
Recall de críticas: 0.233



### 7.4 Comparación de los modelos

Después de probar los tres modelos, he visto que la Regresión Logística ha conseguido los mejores resultados en los datos de validación.

Ha obtenido un accuracy del 67,03 % y un
F1-macro de 0,4775.

Random Forest ha obtenido un accuracy del
64,83 % y un F1-macro de 0,4162.

La diferencia está en las vulnerabilidades críticas. La Regresión Logística ha detectado el 46,36 %, mientras que el Random Forest ha detectado el 23,30 %.

Por estos resultados, voy a seguir tratando de
mejorando la Regresión Logística.

Aunque los dos modelos mejoran el baseline,
la Regresión Logística parece más útil
porque consigue identificar un mayor
porcentaje de vulnerabilidades críticas.

Aun así, sigue dejando algunas sin detectar,
por lo que todavía no sería adecuado
utilizarla sin supervisión humana.

In [28]:

# Comparación de modelos con resultados reales

resultados = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regresión Logística",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_val, pred_baseline),
        accuracy_score(y_val, pred_logistico),
        accuracy_score(y_val, pred_rf)
    ],
    "F1-macro": [
        f1_score(y_val, pred_baseline,
                 average="macro", zero_division=0),
        f1_score(y_val, pred_logistico,
                 average="macro", zero_division=0),
        f1_score(y_val, pred_rf,
                 average="macro", zero_division=0)
    ],
    "Recall críticas": [
        recall_score(y_val == "CRITICAL",
                     pred_baseline == "CRITICAL",
                     zero_division=0),
        recall_score(y_val == "CRITICAL",
                     pred_logistico == "CRITICAL",
                     zero_division=0),
        recall_score(y_val == "CRITICAL",
                     pred_rf == "CRITICAL",
                     zero_division=0)
    ]
})

display(resultados.round(4))


,Modelo,Accuracy,F1-macro,Recall críticas
0,Baseline,0.4805,0.1623,0.0000
1,Regresión Logística,0.6703,0.4775,0.4636
2,Random Forest,0.6483,0.4162,0.2330



## 8. Métricas y errores del modelo

Aunque he conseguido mejores resultados que
los otros modelos, todavía no identifico
correctamente todas las vulnerabilidades.

Para analizarlo, voy a utilizar una matriz
de confusión, que me permitirá ver qué gravedades predice bien y cuáles confunde.

Me interesa especialmente comprobar cuántas
vulnerabilidades críticas se clasifican
como si fueran menos graves.

Esto es indispensable muy importante porque, en una empresa, un error de estos podría hacer que un problema de seguridad importante no se revisara a tiempo, y justo eso es lo que queremos evitar.

In [29]:

# Matriz de confusión de la Regresión Logística

from sklearn.metrics import confusion_matrix
import plotly.express as px

clases = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

matriz = confusion_matrix(
    val["gravedad"],
    pred_logistico,
    labels=clases
)

fig = px.imshow(
    matriz,
    x=clases,
    y=clases,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Matriz de confusión - Regresión Logística",
    labels={
        "x": "Gravedad predicha",
        "y": "Gravedad real",
        "color": "Número de CVE"
    }
)

fig.update_layout(
    width=750,
    height=600
)

fig.show()

print("Matriz de confusión:")
display(
    pd.DataFrame(
        matriz,
        index=clases,
        columns=clases
    )
)


Matriz de confusión:


,LOW,MEDIUM,HIGH,CRITICAL
LOW,1,134,36,0
MEDIUM,2,2546,504,59
HIGH,1,788,1417,176
CRITICAL,0,162,273,376



### 8.1 Análisis de la matriz de confusión

Después de ver la matriz de confusión,
he visto que mi modelo ha conseguido identificar correctamente 376 de las 811 vulnerabilidades críticas.

Sin embargo, hay 435 vulnerabilidades críticas
que no ha clasificado bien.

De ellas, 273 se han clasificado como altas
y 162 como medias.

Esto significa que el modelo todavía tiene
errores importantes, aunque haya mejorado
respecto al baseline.

También he visto que le cuesta bastante identificar las vulnerabilidades de gravedad baja.

Si una vulnerabilidad crítica se clasifica
como media, podría recibir menos atención
de la que necesita.


In [30]:

# Analizar los errores en vulnerabilidades críticas

criticas_reales = val["gravedad"] == "CRITICAL"

total_criticas = criticas_reales.sum()

detectadas = (
    criticas_reales
    & (pred_logistico == "CRITICAL")
).sum()

no_detectadas = total_criticas - detectadas

criticas_como_medias = (
    criticas_reales
    & (pred_logistico == "MEDIUM")
).sum()

criticas_como_altas = (
    criticas_reales
    & (pred_logistico == "HIGH")
).sum()

print("Críticas reales:", total_criticas)
print("Críticas detectadas:", detectadas)
print("Críticas no detectadas:", no_detectadas)
print("Críticas clasificadas como MEDIUM:", criticas_como_medias)
print("Críticas clasificadas como HIGH:", criticas_como_altas)

print(
    "Porcentaje de críticas detectadas:",
    round(detectadas / total_criticas * 100, 2),
    "%"
)


Críticas reales: 811
Críticas detectadas: 376
Críticas no detectadas: 435
Críticas clasificadas como MEDIUM: 162
Críticas clasificadas como HIGH: 273
Porcentaje de críticas detectadas: 46.36 %



## 9. Validación y mejora del modelo

Después de analizar los resultados, he visto
que la Regresión Logística es el modelo que mejor ha funcionado hasta ahora.

Sin embargo, todavía deja sin detectar
bastantes vulnerabilidades críticas.

Por eso, en este apartado voy a intentar probar diferentes configuraciones del modelo para intentar mejorar sus resultados.

Es importante comprobar que funciona
con diferentes datos y que no deja pasar
demasiadas vulnerabilidades críticas.

In [31]:

# Validación respetando el orden temporal

from sklearn.model_selection import TimeSeriesSplit

cv_temporal = TimeSeriesSplit(n_splits=3)

X_cv = train["descripcion_limpia"]
y_cv = train["gravedad"]

print("Divisiones para validación cruzada:\n")

for numero, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_cv), start=1
):
    fecha_train = train.iloc[idx_train]["fecha"]
    fecha_val = train.iloc[idx_val]["fecha"]

    print(f"División {numero}")
    print("Registros de entrenamiento:", len(idx_train))
    print("Registros de validación:", len(idx_val))
    print("Última fecha de entrenamiento:", fecha_train.max())
    print("Primera fecha de validación:", fecha_val.min())
    print("Orden temporal correcto:",
          fecha_train.max() < fecha_val.min())
    print("-" * 40)


Divisiones para validación cruzada:

División 1
Registros de entrenamiento: 7555
Registros de validación: 7552
Última fecha de entrenamiento: 2025-04-07 10:15:15.153000
Primera fecha de validación: 2025-04-07 10:15:16.700000
Orden temporal correcto: True
----------------------------------------
División 2
Registros de entrenamiento: 15107
Registros de validación: 7552
Última fecha de entrenamiento: 2025-06-13 13:15:21.187000
Primera fecha de validación: 2025-06-13 13:15:21.710000
Orden temporal correcto: True
----------------------------------------
División 3
Registros de entrenamiento: 22659
Registros de validación: 7552
Última fecha de entrenamiento: 2025-08-22 08:15:31.017000
Primera fecha de validación: 2025-08-22 09:15:34.410000
Orden temporal correcto: True
----------------------------------------



### 9.1 Mejora de la Regresión Logística

Ahora voy a probar diferentes configuraciones
de la Regresión Logística para intentar
mejorar sus resultados.

Para hacerlo, utilizaré GridSearchCV,
que permite comparar varias combinaciones
de parámetros mediante la validación.

Me interesa especialmente mejorar el F1-macro,
porque tengo cuatro categorías de gravedad
y algunas aparecen mucho menos que otras.

No utilizaré los datos de prueba final
para elegir la configuración, porque quiero
guardarlos para adelante ya evaluar de manera definitiva.

In [32]:

# Mejora con GridSearchCV

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline_grid = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=8000,
        min_df=3,
        stop_words="english"
    )),
    ("clasificador", LogisticRegression(
        max_iter=500,
        random_state=42
    ))
])

parametros = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clasificador__C": [0.1, 1, 5],
    "clasificador__class_weight": [None, "balanced"]
}

cv_temporal = TimeSeriesSplit(n_splits=3)

busqueda = GridSearchCV(
    estimator=pipeline_grid,
    param_grid=parametros,
    scoring="f1_macro",
    cv=cv_temporal,
    n_jobs=2,
    verbose=1
)

busqueda.fit(
    train["descripcion_limpia"],
    train["gravedad"]
)

print("Mejores parámetros:")
print(busqueda.best_params_)

print("\nMejor F1-macro en validación cruzada:")
print(round(busqueda.best_score_, 4))


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Mejores parámetros:
{'clasificador__C': 5, 'clasificador__class_weight': 'balanced', 'tfidf__ngram_range': (1, 2)}

Mejor F1-macro en validación cruzada:
0.5269


In [33]:

# Mirar el modelo mejorado con datos de validación

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score
)

modelo_mejorado = busqueda.best_estimator_

pred_mejorado = modelo_mejorado.predict(
    val["descripcion_limpia"]
)

accuracy_mejorado = accuracy_score(
    val["gravedad"], pred_mejorado
)

f1_mejorado = f1_score(
    val["gravedad"],
    pred_mejorado,
    average="macro",
    zero_division=0
)

recall_mejorado = recall_score(
    val["gravedad"] == "CRITICAL",
    pred_mejorado == "CRITICAL",
    zero_division=0
)

print("Accuracy:", round(accuracy_mejorado, 4))
print("F1-macro:", round(f1_mejorado, 4))
print("Recall de críticas:", round(recall_mejorado, 4))


Accuracy: 0.627
F1-macro: 0.5044
Recall de críticas: 0.6215



### 9.2 Resultados de la mejora del modelo

Después de probar diferentes configuraciones,
he conseguido mejorar la detección de vulnerabilidades críticas.

El modelo original detectaba el 46,36 %,
mientras que el modelo mejorado ha conseguido
detectar el 62,15 %.

También ha mejorado el F1-macro, pasando
de 0,4775 a 0,5044.

Pero he visto que, el accuracy ha bajado del
67,03 % al 62,70 %.

Osea que el modelo ahora identifica más vulnerabilidades críticas, aunque comete
más errores en otras categorías.

In [34]:

# Comparar la detección de críticas

from sklearn.metrics import confusion_matrix

clases = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

matriz_original = confusion_matrix(
    val["gravedad"],
    pred_logistico,
    labels=clases
)

matriz_mejorada = confusion_matrix(
    val["gravedad"],
    pred_mejorado,
    labels=clases
)

comparacion_criticas = pd.DataFrame({
    "Resultado": [
        "Críticas detectadas",
        "Críticas como HIGH",
        "Críticas como MEDIUM",
        "Críticas como LOW"
    ],
    "Modelo original": [
        matriz_original[3, 3],
        matriz_original[3, 2],
        matriz_original[3, 1],
        matriz_original[3, 0]
    ],
    "Modelo mejorado": [
        matriz_mejorada[3, 3],
        matriz_mejorada[3, 2],
        matriz_mejorada[3, 1],
        matriz_mejorada[3, 0]
    ]
})

display(comparacion_criticas)


,Resultado,Modelo original,Modelo mejorado
0,Críticas detectadas,376,504
1,Críticas como HIGH,273,192
2,Críticas como MEDIUM,162,89
3,Críticas como LOW,0,26



### 9.3 Comparación de los errores

Después de comparar los dos modelos, he visto
que el mejorado consigue detectar más
vulnerabilidades críticas.

El modelo original detectaba 376, mientras
que el de ahora ha conseguido identificar 504.

Esto supone 128 vulnerabilidades críticas
más detectadas correctamente.

Sin embargo, también he encontrado otro problema: el modelo mejorado ha clasificado 26
vulnerabilidades críticas como si fueran bajas.


In [35]:

# Identificar vulnerabilidades críticas clasificadas como LOW

errores_graves = val.loc[
    (val["gravedad"] == "CRITICAL")
    & (pred_mejorado == "LOW"),
    ["cve_id", "fecha", "gravedad"]
].copy()

errores_graves["prediccion"] = "LOW"

print("Total de errores CRITICAL -> LOW:", len(errores_graves))

display(errores_graves.head(10))


Total de errores CRITICAL -> LOW: 26


,cve_id,fecha,gravedad,prediccion
30978,CVE-2025-11833,2025-11-01 04:15:42.660,CRITICAL,LOW
31062,CVE-2025-12642,2025-11-03 20:17:06.410,CRITICAL,LOW
31302,CVE-2025-47776,2025-11-04 21:15:37.367,CRITICAL,LOW
31424,CVE-2025-62596,2025-11-06 00:15:37.817,CRITICAL,LOW
31426,CVE-2025-64163,2025-11-06 00:15:38.150,CRITICAL,LOW
31825,CVE-2025-12925,2025-11-10 02:15:34.917,CRITICAL,LOW
32322,CVE-2025-63289,2025-11-12 16:15:36.770,CRITICAL,LOW
32323,CVE-2025-63353,2025-11-12 16:15:36.877,CRITICAL,LOW
32921,CVE-2025-54321,2025-11-18 19:15:49.043,CRITICAL,LOW
33037,CVE-2025-65025,2025-11-19 18:15:49.840,CRITICAL,LOW



### 9.4 Revisión de errores importantes

He revisado algunos casos en los que mi modelo
ha clasificado una vulnerabilidad crítica como baja.

Por ejemplo, he encontrado vulnerabilidades
como CVE-2025-12925 y CVE-2025-63289,
que si que son críticas, pero el modelo
las ha clasificado como LOW.

Esto me dice que todavía puede cometer
errores importantes y que no siempre
debemos confiar en sus predicciones.

Aunque el modelo mejorado detecta más
vulnerabilidades críticas que el original,
también hay casos en los que se vuelve a equivocar de una forma que podría ser peligrosa.



## 10. Interpretación de las predicciones

No quiero fijarme solamente en el porcentaje
de aciertos, sino también en los motivos
por los que realiza determinadas predicciones.

Por eso voy a analizar qué palabras
de las descripciones tienen más peso
en cada nivel de gravedad.

Esto me deja comprobar si el modelo
está aprendiendo patrones relacionados con las vulnerabilidades o si está utilizando cosas que podrían dar resultados poco fiables.

In [36]:

# Palabras con mayor valor en cada categoría

import pandas as pd

tfidf = modelo_mejorado.named_steps["tfidf"]
clasificador = modelo_mejorado.named_steps["clasificador"]

palabras = tfidf.get_feature_names_out()
coeficientes = clasificador.coef_

for i, clase in enumerate(clasificador.classes_):
    indices = coeficientes[i].argsort()[-10:][::-1]

    tabla = pd.DataFrame({
        "Palabra": palabras[indices],
        "Peso": coeficientes[i][indices]
    })

    print(f"\nPalabras asociadas a {clase}:")
    display(tabla.round(3))



Palabras asociadas a CRITICAL:


,Palabra,Peso
0,takeover,5.450
1,remote code,4.360
2,unauthenticated,4.337
3,azure,4.320
4,deserialization,4.210
5,data tampering,4.201
6,update delete,3.966
7,confidentiality integrity,3.915
8,sites,3.869
9,entity reference,3.867



Palabras asociadas a HIGH:


,Palabra,Peso
0,allows reflected,10.102
1,xss issue,7.897
2,reflected xss,5.442
3,search path,4.881
4,local privilege,4.601
5,reflected,4.186
6,csrf vulnerability,3.974
7,able gain,3.959
8,uaf,3.913
9,attackers append,3.817



Palabras asociadas a LOW:


,Palabra,Peso
0,problematic,8.290
1,classified problematic,5.606
2,high privileged,5.339
3,auth,5.195
4,session tokens,5.167
5,photos,5.162
6,partial,5.120
7,potentially leading,5.044
8,reported,4.840
9,low impact,4.657



Palabras asociadas a MEDIUM:


,Palabra,Peso
0,forgery issue,6.388
1,xss,5.502
2,allows cross,4.625
3,authenticated attackers,3.859
4,12 allows,3.829
5,vulnerability high,3.534
6,leads cross,3.327
7,local,3.133
8,served,3.105
9,stores,3.062


In [37]:

# Revisar las palabras asociadas a CRITICAL

import numpy as np
import pandas as pd

indice_critico = list(
    clasificador.classes_
).index("CRITICAL")

pesos_criticos = coeficientes[indice_critico]

indices = np.argsort(pesos_criticos)[-15:][::-1]

tabla_criticas = pd.DataFrame({
    "Palabra": palabras[indices],
    "Peso": pesos_criticos[indices]
})

print("Palabras más asociadas a CRITICAL:")
display(tabla_criticas.round(3))


Palabras más asociadas a CRITICAL:


,Palabra,Peso
0,takeover,5.450
1,remote code,4.360
2,unauthenticated,4.337
3,azure,4.320
4,deserialization,4.210
5,data tampering,4.201
6,update delete,3.966
7,confidentiality integrity,3.915
8,sites,3.869
9,entity reference,3.867



### 10.1 Palabras que utiliza el modelo

Por ejemplo, aparecen términos como
"unauthenticated", "deserialization"
y "data tampering".

Estas palabras hablan sobre el funcionamiento de la información, sobre cómo funciona una vulnerabilidad y qué consecuencias podría tener.

Aunque, también aparecen palabras como "azure", que no indican por sí solas
que una vulnerabilidad sea crítica.


In [38]:

# Revisar tres predicciones de manera individual

ejemplos = val[
    ["cve_id", "descripcion_limpia", "gravedad"]
].copy()

ejemplos["prediccion"] = pred_mejorado

# Seleccionar ejemplos de distintos resultados
casos = pd.concat([
    ejemplos[
        (ejemplos["gravedad"] == "CRITICAL")
        & (ejemplos["prediccion"] == "CRITICAL")
    ].head(1),

    ejemplos[
        (ejemplos["gravedad"] == "CRITICAL")
        & (ejemplos["prediccion"] == "LOW")
    ].head(1),

    ejemplos[
        (ejemplos["gravedad"] == "MEDIUM")
        & (ejemplos["prediccion"] == "MEDIUM")
    ].head(1)
])

display(
    casos[
        ["cve_id", "gravedad", "prediccion",
         "descripcion_limpia"]
    ]
)


,cve_id,gravedad,prediccion,descripcion_limpia
30261,CVE-2025-62959,CRITICAL,CRITICAL,Improper Control of Generation of Code ('Code ...
30978,CVE-2025-11833,CRITICAL,LOW,The Post SMTP – Complete SMTP Solution with Lo...
30211,CVE-2025-62904,MEDIUM,MEDIUM,Improper Neutralization of Input During Web Pa...



### 10.2 Ejemplos de predicciones del modelo

He seleccionado tres vulnerabilidades reales
para comprobar cómo funciona mi modelo.

La primera es CVE-2025-62959, que tiene
una gravedad crítica y el modelo ha
clasificado bien como CRITICAL.

La segunda es CVE-2025-11833, que también
es crítica, pero el modelo la ha clasificado
mal como LOW.

Por último, CVE-2025-62904 tiene una gravedad
media y el modelo la ha identificado bien.

Aunque el modelo puede ayudar a organizar
las vulnerabilidades, no siempre acierta.


In [39]:

# Explicar qué palabras afectan en cada predicción

import numpy as np
import pandas as pd

vectorizador = modelo_mejorado.named_steps["tfidf"]
clasificador = modelo_mejorado.named_steps["clasificador"]

for _, caso in casos.iterrows():

    texto = caso["descripcion_limpia"]
    prediccion = caso["prediccion"]

    # Transformar el texto con el TF-IDF ya entrenado
    vector = vectorizador.transform([texto])

    # Obtener los coeficientes de la clase predicha
    indice_clase = list(
        clasificador.classes_
    ).index(prediccion)

    pesos = clasificador.coef_[indice_clase]

    # Calcular contribuciones de las palabras presentes
    contribuciones = (
        vector.multiply(pesos)
        .toarray()
        .ravel()
    )

    indices = np.argsort(contribuciones)[-5:][::-1]

    tabla = pd.DataFrame({
        "Palabra": vectorizador.get_feature_names_out()[indices],
        "Contribución": contribuciones[indices]
    })

    tabla = tabla[tabla["Contribución"] > 0]

    print("\nCVE:", caso["cve_id"])
    print("Gravedad real:", caso["gravedad"])
    print("Predicción:", prediccion)
    display(tabla.round(4))



CVE: CVE-2025-62959
Gravedad real: CRITICAL
Predicción: CRITICAL


,Palabra,Contribución
0,remote code,0.6599
1,inclusion,0.6299
2,code,0.4998
3,inclusion issue,0.3880
4,remote,0.3070



CVE: CVE-2025-11833
Gravedad real: CRITICAL
Predicción: LOW


,Palabra,Contribución
0,smtp,2.2793
1,links,0.3136
2,logged,0.2987
3,reset,0.2736
4,including makes,0.2637



CVE: CVE-2025-62904
Gravedad real: MEDIUM
Predicción: MEDIUM


,Palabra,Contribución
0,wp,0.9472
1,xss,0.7554
2,allows stored,0.4418
3,stored xss,0.4031
4,scripting vulnerability,0.3929



### 10.3 Interpretación de las palabras

Por ejemplo, en la vulnerabilidad
CVE-2025-62904, que se ha clasificado
bien como MEDIUM, aparecen palabras como "xss", "stored xss" y "wp".

Estas palabras han ayudado a que el
modelo eligiera la categoría MEDIUM.


In [40]:

# Explicar el error de CVE-2025-11833

caso = casos[
    casos["cve_id"] == "CVE-2025-11833"
].iloc[0]

vector = vectorizador.transform(
    [caso["descripcion_limpia"]]
)

indice = list(clasificador.classes_).index("LOW")

contribuciones = (
    vector.multiply(clasificador.coef_[indice])
    .toarray()
    .ravel()
)

indices = np.argsort(contribuciones)[-5:][::-1]

tabla_error = pd.DataFrame({
    "Palabra": vectorizador.get_feature_names_out()[indices],
    "Contribución": contribuciones[indices]
})

print("Vulnerabilidad:", caso["cve_id"])
print("Gravedad real:", caso["gravedad"])
print("Predicción del modelo:", caso["prediccion"])

display(tabla_error.round(4))


Vulnerabilidad: CVE-2025-11833
Gravedad real: CRITICAL
Predicción del modelo: LOW


,Palabra,Contribución
0,smtp,2.2793
1,links,0.3136
2,logged,0.2987
3,reset,0.2736
4,including makes,0.2637



### 10.4 Análisis de una predicción que no esta bien

Al analizar las palabras que han estado
en la predicción, he visto que destaca
"SMTP", con una contribución de 2,2793.

También aparecen otras palabras como
"links", "logged" y "reset".

Por ejemplo, SMTP es un protocolo de correo,
pero eso no significa que una vulnerabilidad
relacionada con él tenga que ser poco grave.

Este ejemplo demuestra que el modelo puede
equivocarse al interpretar la información
de una vulnerabilidad.

Por eso, considero importante revisar las
predicciones, especialmente cuando una
vulnerabilidad aparece como baja.


In [41]:

# Comprobar si los datos siguen cargados

variables = ["train", "val", "test", "modelo_mejorado", "pred_mejorado"]

for nombre in variables:
    if nombre in globals():
        print(nombre, ": Disponible")
    else:
        print(nombre, ": No disponible")


train : Disponible
val : Disponible
test : Disponible
modelo_mejorado : Disponible
pred_mejorado : Disponible



### 10.5 Comparación de errores por meses

Si funciona bien en unos meses, pero falla
mucho en otros, significaría que sus
resultados no son igual de fiables siempre.

Esto es importante porque las vulnerabilidades
cambian con el tiempo y una empresa necesita
poder analizar también problemas que aparezcan nuevos.


In [42]:

# Comparar el rendimiento del modelo por meses

evaluacion_mensual = val[
    ["fecha", "gravedad"]
].copy()

evaluacion_mensual["prediccion"] = pred_mejorado

evaluacion_mensual["mes"] = (
    evaluacion_mensual["fecha"]
    .dt.to_period("M")
    .astype(str)
)

evaluacion_mensual["acierto"] = (
    evaluacion_mensual["gravedad"]
    == evaluacion_mensual["prediccion"]
)

resumen_mensual = (
    evaluacion_mensual
    .groupby("mes")
    .agg(
        total=("acierto", "size"),
        accuracy=("acierto", "mean")
    )
    .reset_index()
)

resumen_mensual["accuracy"] *= 100

display(resumen_mensual.round(2))


,mes,total,accuracy
0,2025-10,761,63.60
1,2025-11,2855,59.33
2,2025-12,2859,65.83



### 10.6 Resultados por meses

He comparado los resultados del modelo
en diferentes meses de publicación.

En octubre de 2025 ha conseguido un
63,60 % de aciertos, en noviembre un
59,33 % y en diciembre un 65,83 %.

He visto que el modelo no funciona exactamente
igual en todos los meses.

El peor resultado aparece en noviembre,
pero que el mejor se obtiene en diciembre.

Una empresa debería revisar el funcionamiento
del modelo con el paso del tiempo.

Si empieza a cometer más errores, sería
necesario comprobar qué está ocurriendo
y valorar si necesita volver a entrenarse de nuevo.


In [43]:

# Calcular el recall de vulnerabilidades críticas por mes

evaluacion_mensual["es_critica"] = (
    evaluacion_mensual["gravedad"] == "CRITICAL"
)

evaluacion_mensual["critica_detectada"] = (
    evaluacion_mensual["es_critica"]
    & (evaluacion_mensual["prediccion"] == "CRITICAL")
)

resumen_criticas_mes = (
    evaluacion_mensual
    .groupby("mes")
    .agg(
        criticas_reales=("es_critica", "sum"),
        criticas_detectadas=("critica_detectada", "sum")
    )
    .reset_index()
)

resumen_criticas_mes["recall_criticas"] = (
    resumen_criticas_mes["criticas_detectadas"]
    / resumen_criticas_mes["criticas_reales"].replace(0, float("nan"))
    * 100
)

display(resumen_criticas_mes.round(2))


,mes,criticas_reales,criticas_detectadas,recall_criticas
0,2025-10,92,61,66.30
1,2025-11,380,253,66.58
2,2025-12,339,190,56.05



### 10.7 Detección de vulnerabilidades críticas por mes

Después de comprobar los resultados por meses, he visto que el modelo no detecta las vulnerabilidades críticas igual de bien en todos los periodos.

En octubre ha identificado el 66,30 %
de las críticas y en noviembre el 66,58%.

Sin embargo, en diciembre el porcentaje
ha bajado al 56,05 %.

Esto me ha hecho ver el porque si en 
diciembre era el mes con más aciertos
generales, pero no el que mejor detectaba
las vulnerabilidades críticas.


## 11. Recomendación para la empresa

Después de analizar los resultados, creo que mi modelo puede ser útil como una
herramienta de apoyo para los equipos
de ciberseguridad.

La Regresión Logística mejorada ha conseguido detectar el 62,15 % de las vulnerabilidades críticas en los datos de validación.

Aunque esto supone una mejora al
modelo original, todavía quedan vulnerabilidades críticas sueltas.

Por eso, no recomendaría utilizarlo para
tomar decisiones automáticas sobre qué
vulnerabilidades se pueden ignorar.

Yo diría que utilizarlo como una
primera ayuda para organizar las revisiones esta bien siempre y cuando se siga manteniendo siempre la supervisión de un especialista.


## 12. Evaluación final

Después de comparar y mejorar los modelos, he decidido escoger la Regresión Logística para hacer la prueba final.

Ahora voy a entrenarla utilizando los datos de entrenamiento y validación juntos.

Después observare cómo funciona con los
datos de prueba, que he guardado hasta ahora.

Quiero comprobar si mantiene unos resultados parecidos a los anteriores y si consigue detectar bien las vulnerabilidades críticas.

In [44]:

# Entrenamiento final 
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, recall_score
import pandas as pd

# Unir entrenamiento y validación
train_final = pd.concat([train, val]).sort_values(
    ["fecha", "cve_id"]
)

# Copiar la configuración elegida sin modificarla
modelo_final = clone(busqueda.best_estimator_)

# Entrenar con todos los datos disponibles antes de prueba
modelo_final.fit(
    train_final["descripcion_limpia"],
    train_final["gravedad"]
)

# Predecir sobre el conjunto de prueba
pred_test = modelo_final.predict(
    test["descripcion_limpia"]
)

# Calcular las métricas finales
accuracy_test = accuracy_score(
    test["gravedad"], pred_test
)

f1_test = f1_score(
    test["gravedad"],
    pred_test,
    average="macro",
    zero_division=0
)

recall_test = recall_score(
    test["gravedad"] == "CRITICAL",
    pred_test == "CRITICAL",
    zero_division=0
)

print("RESULTADOS FINALES")
print("Accuracy:", round(accuracy_test, 4))
print("F1-macro:", round(f1_test, 4))
print("Recall de críticas:", round(recall_test, 4))


RESULTADOS FINALES
Accuracy: 0.6078
F1-macro: 0.507
Recall de críticas: 0.5843



### 12.1 Resultados de la prueba final

Después de entrenar el modelo con los datos de entrenamiento y validación, he hecho la prueba final con los registros que había guardado.

He obtenido un accuracy del 60,78 %,
un F1-macro de 0,5070 y un recall de
vulnerabilidades críticas del 58,43 %.

Estos resultados son parecidos a los
obtenidos antes, aunque la detección de vulnerabilidades críticas ha bajado un poco.

Esto significa que el modelo ha conseguido aprender algunos patrones de las descripciones, pero todavía comete errores importantes.

Mi objetivo era detectar al menos
el 90 % de las vulnerabilidades críticas,
por lo que todavía no lo he logrado.

El modelo podría ayudar a organizar
las vulnerabilidades y señalar posibles
problemas importantes.


In [45]:

# Guardar el modelo final

from pathlib import Path
import joblib

ruta_modelo = Path("model.joblib")

joblib.dump(modelo_final, ruta_modelo)

print("Modelo guardado correctamente")
print("Ruta:", ruta_modelo.resolve())
print(
    "Tamaño:",
    round(ruta_modelo.stat().st_size / (1024 * 1024), 2),
    "MB"
)

# Cargar de nuevo el modelo y comprobarlo
modelo_cargado = joblib.load(ruta_modelo)

predicciones_cargado = modelo_cargado.predict(
    test["descripcion_limpia"].head(5)
)

print("\nPredicciones del modelo cargado:")
print(predicciones_cargado)

print("\nCoinciden con el modelo original:")
print(
    (predicciones_cargado == pred_test[:5]).all()
)


Modelo guardado correctamente
Ruta: /workspaces/BPS-DsciAinara/project/model.joblib
Tamaño: 0.55 MB

Predicciones del modelo cargado:
['CRITICAL' 'CRITICAL' 'CRITICAL' 'HIGH' 'MEDIUM']

Coinciden con el modelo original:
True



### 12.2 Comprobación del modelo

Después de terminar las pruebas, he guardado mi modelo de Regresión Logística en un archivo llamado model.joblib.

El archivo ocupa como 0,55 MB.

Para comprobar que se había guardado bien, lo he vuelto a cargar con Python y he realizado varias predicciones.

Los resultados han justo coincidido con los que había obtenido antes de guardar el modelo.

Esto significa que puedo reutilizarlo sin
tener que entrenarlo de nuevo cada vez.


In [46]:

# Guardar el dataset en formato Parquet

from pathlib import Path

ruta_datos = Path("data")
ruta_datos.mkdir(exist_ok=True)

columnas_exportar = [
    "cve_id",
    "fecha",
    "estado",
    "descripcion",
    "descripcion_limpia",
    "gravedad",
    "cwe"
]

df_exportar = df_limpio[columnas_exportar].copy()

archivo_parquet = ruta_datos / "cve_2025_procesado.parquet"

df_exportar.to_parquet(
    archivo_parquet,
    index=False,
    compression="zstd"
)

print("Archivo generado correctamente")
print("Registros:", len(df_exportar))
print("Columnas:", len(df_exportar.columns))
print(
    "Tamaño:",
    round(
        archivo_parquet.stat().st_size / (1024 * 1024),
        2
    ),
    "MB"
)


Archivo generado correctamente
Registros: 43160
Columnas: 7
Tamaño: 9.27 MB


In [47]:

# Ver el archivo Parquet

import pandas as pd
from pathlib import Path

ruta_parquet = Path("data/cve_2025_procesado.parquet")

df_comprobacion = pd.read_parquet(ruta_parquet)

print("Archivo leído correctamente")
print("Registros:", len(df_comprobacion))
print("Columnas:", len(df_comprobacion.columns))

print("\n¿Coinciden los identificadores CVE?")
print(
    df_comprobacion["cve_id"].equals(
        df_limpio["cve_id"].reset_index(drop=True)
    )
)

print("\n¿Coinciden las categorías de gravedad?")
print(
    df_comprobacion["gravedad"].equals(
        df_limpio["gravedad"].reset_index(drop=True)
    )
)


Archivo leído correctamente
Registros: 43160
Columnas: 7

¿Coinciden los identificadores CVE?
True

¿Coinciden las categorías de gravedad?
True



## 13. Reproducción del proyecto

Para realizar este proyecto he utilizado
el dataset original de vulnerabilidades CVE de la National Vulnerability Database.

El archivo original ocupa 277 MB, por lo que no lo he incluido directamente en GitHub.

Si no que, he preparado un archivo Parquet, he tenido varios problemas al principio para crearlo pero cuando lo hice pues comprobe las 43.160 vulnerabilidades y las columnas necesarias para reproducir el entrenamiento y la evaluación del modelo.

Este archivo me deja trabajar con los datos preparados sin tener que descargar el JSON original.

Por eso, el Parquet no sustituye completamente al dataset original.


In [48]:

# Cargar los datos preparados desde Parquet

import pandas as pd
from pathlib import Path

ruta_parquet = Path("data/cve_2025_procesado.parquet")

datos_reproducibles = pd.read_parquet(ruta_parquet)

datos_reproducibles["fecha"] = pd.to_datetime(
    datos_reproducibles["fecha"]
)

print("Datos cargados desde Parquet")
print("Registros:", len(datos_reproducibles))
print("Columnas:", len(datos_reproducibles.columns))

display(datos_reproducibles.head(3))


Datos cargados desde Parquet
Registros: 43160
Columnas: 7


,cve_id,fecha,estado,descripcion,descripcion_limpia,gravedad,cwe
0,CVE-2025-0168,2025-01-01 14:15:23.590,Analyzed,A vulnerability classified as critical has bee...,A vulnerability classified as has been found i...,HIGH,CWE-74
1,CVE-2025-22214,2025-01-02 04:15:06.277,Deferred,Landray EIS 2001 through 2006 allows Message/f...,Landray EIS 2001 through 2006 allows Message/f...,MEDIUM,CWE-89
2,CVE-2025-0171,2025-01-02 15:15:25.550,Analyzed,"A vulnerability, which was classified as criti...","A vulnerability, which was classified as , was...",HIGH,CWE-74


In [49]:

# Reproducir la división desde Parquet

df_repro = datos_reproducibles.sort_values(
    ["fecha", "cve_id"]
).reset_index(drop=True)

total_repro = len(df_repro)

fin_train_repro = int(total_repro * 0.70)
fin_val_repro = int(total_repro * 0.85)

train_repro = df_repro.iloc[:fin_train_repro].copy()
val_repro = df_repro.iloc[fin_train_repro:fin_val_repro].copy()
test_repro = df_repro.iloc[fin_val_repro:].copy()

print("Entrenamiento:", len(train_repro))
print("Validación:", len(val_repro))
print("Prueba:", len(test_repro))

print("\n¿Coinciden los identificadores con la división original?")
print("Train:", train_repro["cve_id"].tolist() == train["cve_id"].tolist())
print("Val:", val_repro["cve_id"].tolist() == val["cve_id"].tolist())
print("Test:", test_repro["cve_id"].tolist() == test["cve_id"].tolist())


Entrenamiento: 30211
Validación: 6475
Prueba: 6474

¿Coinciden los identificadores con la división original?
Train: True
Val: True
Test: True


In [50]:

# Hacer las predicciones con el modelo guardado

import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, recall_score

ruta_modelo = Path("model.joblib")

modelo_reproducido = joblib.load(ruta_modelo)

pred_repro = modelo_reproducido.predict(
    test_repro["descripcion_limpia"]
)

print("Predicciones realizadas:", len(pred_repro))

print("Accuracy:",
      round(accuracy_score(
          test_repro["gravedad"], pred_repro
      ), 4))

print("F1-macro:",
      round(f1_score(
          test_repro["gravedad"], pred_repro,
          average="macro",
          zero_division=0
      ), 4))

print("Recall de críticas:",
      round(recall_score(
          test_repro["gravedad"] == "CRITICAL",
          pred_repro == "CRITICAL",
          zero_division=0
      ), 4))

print("\n¿Coinciden con las predicciones originales?")
print((pred_repro == pred_test).all())


Predicciones realizadas: 6474
Accuracy: 0.6078
F1-macro: 0.507
Recall de críticas: 0.5843

¿Coinciden con las predicciones originales?
True



### 13.1 Cómo comprobar los resultados del proyecto

Para hacer sencilla la revisión de mi trabajo he creado un archivo llamado reproducir_modelo.py.

Este programa utiliza el dataset preparado en formato Parquet y el modelo guardado en model.joblib.

Al ejecutarlo, he conseguido reproducir
los resultados de la evaluación final:

- Accuracy: 60,78 %
- F1-macro: 0,5070
- Recall de vulnerabilidades críticas: 58,43 %

El programa deja comprobar estos resultados sin tener que ejecutar todas las celdas del notebook ni descargar el JSON original.

Para utilizarlo, hay que instalar las librerías de requirements.txt y ejecutar desde el principio del repositorio:

python project/reproducir_modelo.py


## 14. Bibliografía

**National Institute of Standards and Technology
(NIST)** (s. f.). *National Vulnerability
Database (NVD)*.
https://nvd.nist.gov/

**NIST** (s. f.). *NVD Data Feeds*.
https://nvd.nist.gov/vuln/data-feeds

**Scikit-learn Developers** (s. f.).
*DummyClassifier: documentación oficial*.
https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html

**Scikit-learn Developers** (s. f.).
*GridSearchCV: documentación oficial*.
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

**Pandas Development Team** (s. f.).
*pandas.read_parquet: documentación oficial*.
https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html


**Ortega, M.** (2026). *Presentaciones de Material de la asignatura, curso 2026–2027. Google Classroom.

**Ortega, M.** (2026). *Apuntes y contenidos teóricos. Google Classroom.


